In [2]:
# inlegalbert_bilstm_mha_crf_proto_v3_fixed.py  (PROTOTYPICAL LEARNING — BUG FIXED)
#
# FIX SUMMARY (RuntimeError: inplace operation modified tensor needed for grad):
#
#   ROOT CAUSE:
#     In PrototypeBank.update_prototypes(), the line:
#         self.prototypes[k] = decay * self.prototypes[k] + (1-decay) * batch_mean
#     performs an in-place __setitem__ on a buffer that PyTorch has already
#     included in the autograd graph (via PrototypicalAttention which reads
#     self.prototypes). Even though @torch.no_grad() wraps the function,
#     the in-place index-assignment mutates the underlying storage that
#     backward() still needs at version 0, but finds at version N.
#
#   FIXES APPLIED:
#     1. PrototypeBank.update_prototypes():
#        Changed  self.prototypes[k] = ...
#        to       self.prototypes.data[k] = ...
#        .data bypasses autograd entirely — no version counter bump,
#        no graph modification.
#
#     2. InLegalBERT_BiLSTM_MHA_Proto_CRF.forward():
#        prototypes = self.proto_bank.get_prototypes()
#        → changed to:
#        prototypes = self.proto_bank.get_prototypes().detach()
#        This prevents gradients from flowing INTO the prototype buffer
#        through PrototypicalAttention, which eliminates the second
#        source of the same error (gradient through a modified tensor).
#
#     3. PrototypicalAttention.forward():
#        emissions input already detached at call site; kept as-is.
#        Added .detach() guard inside forward as extra safety.
#
#     4. prototypical_loss():
#        prototypes passed in as .detach() copy so loss gradients
#        flow only through embeddings, not back into the buffer.
#
#   All other logic, hyperparameters, and architecture are unchanged.

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_proto_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 80
BERT_LR         = 1e-5
HEAD_LR         = 2e-4
WEIGHT_DECAY    = 0.1
GRAD_CLIP       = 1.0
DROPOUT         = 0.5

BERT_FREEZE_LAYERS  = 10
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1

MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1

AUX_CE_WEIGHT    = 0.3
LABEL_SMOOTHING  = 0.1

SWA_START_FRAC   = 0.80
SWA_LR           = 5e-5

ES_PATIENCE      = 8
ES_MIN_DELTA     = 1e-4

GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_RATIO    = 0.05
RARE_THRESHOLD  = 0.05

# ── Prototypical Learning Hyperparameters ──────────────────
PROTO_WEIGHT        = 0.3
PROTO_TEMPERATURE   = 0.1
PROTO_EMA_COMMON    = 0.99
PROTO_EMA_RARE      = 0.90
PROTO_RARE_WEIGHT   = 3.0
PROTO_GATE_INIT     = 0.0    # sigmoid(0.0) = 0.5

# ── Label taxonomy ─────────────────────────────────────────
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":            "InLegalBERT Encoder",
        "sent_bilstm":     "Sentence BiLSTM",
        "mha_pooling":     "MHA Pooling",
        "ctx_bilstm":      "Context BiLSTM",
        "proto_bank":      "Prototype Bank",
        "proto_attention": "Prototypical Attention",
        "classifier":      "Classifier Head",
        "crf":             "CRF",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_trainable + total_frozen,
    })

    print("\n" + "=" * 74)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + Proto + CRF)")
    print("=" * 74)
    print(f"  {'Component':<32} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 74)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 74)
        print(f"  {r['Component']:<32} "
              f"{r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} "
              f"{r['Total Params']:>12,}")
    print("=" * 74)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MHA POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query     = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        w = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            w = w.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        w = self.attn_drop(F.softmax(w, dim=-1))
        return self.out_proj(torch.matmul(w, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# P1. PROTOTYPE BANK  — FIXED
# ═══════════════════════════════════════════════════════════
class PrototypeBank(nn.Module):
    """
    Maintains one EMA prototype per rhetorical class.

    FIX: update_prototypes() now uses self.prototypes.data[k] = ...
    instead of self.prototypes[k] = ...

    self.prototypes[k] = X  →  in-place __setitem__ on the storage
    that autograd tracks; bumps the version counter; breaks backward().

    self.prototypes.data[k] = X  →  writes to the raw storage directly,
    bypassing autograd entirely; version counter is NOT bumped; backward()
    sees the tensor as unmodified.
    """

    def __init__(
        self,
        embed_dim:  int,
        num_labels: int,
        rare_ids:   list = None,
        ema_common: float = PROTO_EMA_COMMON,
        ema_rare:   float = PROTO_EMA_RARE,
    ):
        super().__init__()
        self.embed_dim  = embed_dim
        self.num_labels = num_labels
        self.rare_ids   = set(rare_ids) if rare_ids else set()
        self.ema_common = ema_common
        self.ema_rare   = ema_rare

        protos = torch.empty(num_labels, embed_dim)
        nn.init.xavier_uniform_(protos)
        self.register_buffer("prototypes", protos)

        self.register_buffer(
            "update_count",
            torch.zeros(num_labels, dtype=torch.long)
        )

        ema_decays = torch.full((num_labels,), fill_value=ema_common)
        for rid in self.rare_ids:
            ema_decays[rid] = ema_rare
        self.register_buffer("ema_decays", ema_decays)

    @torch.no_grad()
    def update_prototypes(self, embeddings: torch.Tensor, labels: torch.Tensor):
        """
        EMA-update prototypes using .data assignment to avoid
        mutating the autograd graph.

        KEY FIX:
            BEFORE (broken):  self.prototypes[k] = new_val
            AFTER  (fixed):   self.prototypes.data[k] = new_val

        self.prototypes.data accesses the underlying Tensor storage
        without touching the autograd metadata. This means:
          - No version counter increment on self.prototypes
          - backward() can still read the "original" prototype values
            that were used in the forward pass
          - The physical memory IS updated, so the next forward() call
            uses the freshly EMA-updated values
        """
        for k in range(self.num_labels):
            mask_k = (labels == k)
            if not mask_k.any():
                continue

            batch_mean = embeddings[mask_k].mean(dim=0)   # (embed_dim,)

            decay = self.ema_decays[k].item()

            # ── FIX: use .data to bypass autograd version tracking ──
            self.prototypes.data[k] = (
                decay * self.prototypes.data[k]
                + (1.0 - decay) * batch_mean
            )
            self.update_count[k] += 1

    def get_prototypes(self) -> torch.Tensor:
        """
        Return a DETACHED copy of the prototype matrix.

        FIX: Always return .detach() so that:
          1. PrototypicalAttention cannot accidentally backprop INTO
             the prototype buffer through W_q/W_k projections.
          2. prototypical_loss() gradients flow only through the
             sentence embeddings, not back to the buffer.

        The prototypes are updated via EMA only (not gradients),
        so detaching is semantically correct.
        """
        return self.prototypes.detach()   # (NUM_LABELS, embed_dim)

    def get_prototypes_for_loss(self) -> torch.Tensor:
        """Alias — same as get_prototypes(), explicit for readability."""
        return self.prototypes.detach()

    def get_prototype_stats(self) -> dict:
        stats = {}
        for k in range(self.num_labels):
            stats[id2label[k]] = {
                "update_count": int(self.update_count[k].item()),
                "proto_norm":   float(self.prototypes[k].norm().item()),
            }
        return stats


# ═══════════════════════════════════════════════════════════
# P2. PROTOTYPICAL ATTENTION  — FIXED
# ═══════════════════════════════════════════════════════════
class PrototypicalAttention(nn.Module):
    """
    Enriches ctx_out using class prototypes as attention queries.

    FIX: The prototypes tensor passed in is already .detach()'ed by
    PrototypeBank.get_prototypes(). We add an extra .detach() call
    at the start of forward() as a defensive guard to ensure no
    gradient can flow back into the prototype buffer through this module,
    regardless of how the caller passes the tensor.

    The emissions input is also detached at the call site in forward()
    of the main model, preventing double-gradient through the classifier.
    """

    def __init__(self, embed_dim: int, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.embed_dim  = embed_dim
        self.num_labels = num_labels
        self.scale      = embed_dim ** -0.5

        self.W_q    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj   = nn.Linear(embed_dim, embed_dim, bias=True)
        self.attn_drop  = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.gate       = nn.Parameter(torch.tensor(float(PROTO_GATE_INIT)))

    def forward(
        self,
        ctx_out:    torch.Tensor,   # (B, T, D)
        prototypes: torch.Tensor,   # (NUM_LABELS, D) — already detached
        emissions:  torch.Tensor,   # (B, T, NUM_LABELS) — already detached
        seq_mask:   torch.Tensor,   # (B, T) bool
    ) -> torch.Tensor:

        # ── Defensive detach guard ────────────────────────────
        # Even if the caller forgets to detach, we ensure no gradient
        # flows back into prototype parameters through this module.
        prototypes = prototypes.detach()
        emissions  = emissions.detach()

        B, T, D = ctx_out.shape
        K       = self.num_labels

        Keys   = self.W_k(ctx_out)                              # (B, T, D)
        Values = self.W_v(ctx_out)                              # (B, T, D)

        Queries = self.W_q(prototypes)                          # (K, D)
        Queries = Queries.unsqueeze(0).expand(B, -1, -1)        # (B, K, D)

        Keys_T = Keys.transpose(1, 2)                           # (B, D, T)

        attn_scores  = torch.bmm(Queries, Keys_T) * self.scale  # (B, K, T)
        pad_mask     = ~seq_mask.unsqueeze(1)                   # (B, 1, T)
        attn_scores  = attn_scores.masked_fill(pad_mask, -1e9)
        attn_weights = self.attn_drop(F.softmax(attn_scores, dim=-1))  # (B, K, T)

        proto_contexts = torch.bmm(attn_weights, Values)        # (B, K, D)

        proto_weights = F.softmax(emissions, dim=-1)            # (B, T, K)
        proto_ctx     = torch.bmm(proto_weights, proto_contexts)# (B, T, D)

        proto_ctx = self.layer_norm(self.out_proj(proto_ctx))   # (B, T, D)

        alpha = torch.sigmoid(self.gate)
        out   = alpha * ctx_out + (1.0 - alpha) * proto_ctx     # (B, T, D)
        return out


# ═══════════════════════════════════════════════════════════
# P3. PROTOTYPICAL LOSS  — FIXED
# ═══════════════════════════════════════════════════════════
def prototypical_loss(
    embeddings: torch.Tensor,   # (N, D) — valid ctx_out vectors (with grad)
    labels:     torch.Tensor,   # (N,)
    prototypes: torch.Tensor,   # (K, D) — DETACHED; no grad flows here
    rare_ids:   set,
    temperature: float = PROTO_TEMPERATURE,
    rare_weight: float = PROTO_RARE_WEIGHT,
) -> torch.Tensor:
    """
    ProtoNet-style loss.

    FIX: prototypes must be passed as a detached tensor (no grad).
    Gradients flow only through `embeddings` (the ctx_out vectors),
    which is what we want: the loss teaches the BiLSTM to place
    sentence vectors close to the correct prototype cluster centre.
    The prototypes themselves are updated via EMA, not gradients.

    Passing a non-detached prototype here would create a second backward
    path into the prototype buffer, conflicting with the EMA-only update
    strategy and causing the version-counter error.
    """
    if embeddings.shape[0] == 0:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    # Defensive: ensure no grad through prototypes
    prototypes = prototypes.detach()

    # Squared Euclidean distance: (N, K)
    e_sq  = (embeddings ** 2).sum(dim=1, keepdim=True)           # (N, 1)
    p_sq  = (prototypes ** 2).sum(dim=1, keepdim=True).T         # (1, K)
    cross = torch.mm(embeddings, prototypes.T)                    # (N, K)
    dists = (e_sq + p_sq - 2.0 * cross).clamp(min=0.0)           # (N, K)

    log_probs = F.log_softmax(-dists / temperature, dim=-1)       # (N, K)
    nll       = F.nll_loss(log_probs, labels, reduction="none")   # (N,)

    weights = torch.ones_like(nll)
    for rid in rare_ids:
        weights[labels == rid] = rare_weight

    return (nll * weights).sum() / weights.sum()


# ═══════════════════════════════════════════════════════════
# MODEL  — FIXED
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_Proto_CRF(nn.Module):
    """
    Full pipeline with prototypical learning.

    KEY FIXES in forward():
      1. prototypes = self.proto_bank.get_prototypes()
         → get_prototypes() now returns .detach() — no grad into buffer.

      2. proto_attention(..., emissions=prelim_emissions.detach(), ...)
         → Already present in original; kept + PrototypicalAttention
           now also internally detaches as defensive guard.

      3. prototypical_loss(..., prototypes=prototypes, ...)
         → prototypes is already detached from step 1.

      4. update_prototypes() called AFTER loss.backward() completes
         in Trainer.train() — moved OUT of forward() to avoid
         any possibility of in-place mutation during the forward pass.
         The model receives update_proto=True flag; the actual EMA
         call is now deferred to after the backward step.

    IMPORTANT CHANGE to training loop (see Trainer.train):
      The EMA update is now called explicitly AFTER loss.backward()
      and optimizer.step(), not inside forward(). This guarantees
      the graph is fully consumed before any buffer mutation occurs.
    """

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = None,
    ):
        super().__init__()
        self.rare_ids    = set(rare_ids) if rare_ids else set()
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim    = self.bert.config.hidden_size
        self.dropout     = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

        self.proto_bank = PrototypeBank(
            embed_dim  = self.ctx_out_dim,
            num_labels = num_labels,
            rare_ids   = rare_ids,
        )

        self.proto_attention = PrototypicalAttention(
            embed_dim  = self.ctx_out_dim,
            num_labels = num_labels,
            dropout    = mha_dropout,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N       = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid      = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)
        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_ctx_out(self, input_ids, attention_mask, token_type_ids, lengths=None):
        """Extract ctx_out for use in EMA update (called after backward)."""
        with torch.no_grad():
            sent_vecs = self.encode_sentences(
                input_ids, attention_mask, token_type_ids, lengths=lengths
            )
            sent_vecs = self.dropout(sent_vecs)
            if lengths is not None:
                packed = nn.utils.rnn.pack_padded_sequence(
                    sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
                )
                packed_out, _ = self.ctx_bilstm(packed)
                ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                    packed_out, batch_first=True
                )
            else:
                ctx_out, _ = self.ctx_bilstm(sent_vecs)
        return ctx_out  # (B, T, ctx_out_dim)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
        update_proto:   bool         = False,  # kept for API compat; EMA now external
    ):
        # ── 1. Sentence encoding ──────────────────────────────
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        # ── 2. Context BiLSTM ─────────────────────────────────
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out = self.dropout(ctx_out)   # (B, T, 128)

        # ── 3. Preliminary emissions (detached for attention) ─
        prelim_emissions = self.classifier(ctx_out)
        prelim_emissions = torch.nan_to_num(
            prelim_emissions, nan=0.0, posinf=1e4, neginf=-1e4
        )

        # ── 4. Sequence validity mask ──────────────────────────
        if lengths is not None:
            B, T, _ = prelim_emissions.shape
            seq_mask = torch.zeros(B, T, dtype=torch.bool,
                                   device=prelim_emissions.device)
            for i, l in enumerate(lengths):
                seq_mask[i, :l] = True
        elif labels is not None:
            seq_mask = (labels != -100)
        else:
            seq_mask = torch.ones(prelim_emissions.shape[:2],
                                  dtype=torch.bool,
                                  device=prelim_emissions.device)

        # ── 5. Prototypical Attention ─────────────────────────
        # FIX: get_prototypes() returns .detach() — no grad into buffer
        prototypes = self.proto_bank.get_prototypes()   # (K, 128) detached

        proto_out = self.proto_attention(
            ctx_out    = ctx_out,
            prototypes = prototypes,              # already detached
            emissions  = prelim_emissions.detach(),  # detach: no double grad
            seq_mask   = seq_mask,
        )   # (B, T, 128)

        # ── 6. Final emissions ─────────────────────────────────
        final_emissions = self.classifier(proto_out)
        final_emissions = torch.nan_to_num(
            final_emissions, nan=0.0, posinf=1e4, neginf=-1e4
        )

        # ── 7. Training losses ────────────────────────────────
        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(
                final_emissions, safe_labels, mask=seq_mask, reduction="mean"
            )

            B2, T2, C = final_emissions.shape
            ce_loss = nn.functional.cross_entropy(
                final_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
                label_smoothing=LABEL_SMOOTHING,
                ignore_index=-100,
                reduction="mean",
            )

            # Prototypical loss on ctx_out (pre-attention, with grad)
            flat_mask_1d = seq_mask.reshape(-1)
            flat_ctx     = ctx_out.reshape(-1, self.ctx_out_dim)
            flat_labels  = labels.reshape(-1)
            valid_ctx    = flat_ctx[flat_mask_1d]      # (N_valid, 128) — has grad
            valid_labels = flat_labels[flat_mask_1d]   # (N_valid,)

            proto_loss = prototypical_loss(
                embeddings  = valid_ctx,
                labels      = valid_labels,
                prototypes  = prototypes,              # already detached
                rare_ids    = self.rare_ids,
                temperature = PROTO_TEMPERATURE,
                rare_weight = PROTO_RARE_WEIGHT,
            )

            # NOTE: EMA update is now called from Trainer.train() AFTER
            # backward(), not here. We return valid_ctx and valid_labels
            # so the trainer can call update_prototypes() externally.
            loss = (
                crf_loss
                + AUX_CE_WEIGHT * ce_loss
                + PROTO_WEIGHT  * proto_loss
            )
            return loss, final_emissions, {
                "crf_loss":    crf_loss.item(),
                "ce_loss":     ce_loss.item(),
                "proto_loss":  proto_loss.item(),
                # Pass tensors needed for EMA update (detached clones)
                "_valid_ctx":    valid_ctx.detach().clone(),
                "_valid_labels": valid_labels.detach().clone(),
            }

        # ── 8. Inference ──────────────────────────────────────
        else:
            return self.crf.decode(final_emissions, mask=seq_mask), final_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc          = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER  — FIXED
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        """
        Layer-wise LR decay for BERT + HEAD_LR for non-BERT.
        PrototypeBank is excluded (EMA-only, no gradients).
        """
        param_groups = []

        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
            self.model.proto_attention,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(p for p in m.parameters() if p.requires_grad)
        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                result = m(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                    update_proto=False,
                )
                loss = result[0]
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        swa_model     = AveragedModel(self.model)
        swa_start_ep  = max(1, int(num_epochs * SWA_START_FRAC))
        swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR,
                              anneal_epochs=5, anneal_strategy="cos")
        swa_active    = False
        print(f"📊 SWA starts at epoch {swa_start_ep}/{num_epochs}.")

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None
        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss = running_crf = running_ce = running_proto = 0.0
            n_steps = nan_steps = 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                # Forward pass — update_proto flag kept for API compat
                # but EMA update happens AFTER backward (see below)
                result = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                    update_proto=False,   # EMA deferred to after backward
                )
                loss, _, sub_losses = result

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss_scaled = loss / GRADIENT_ACCUMULATION_STEPS
                loss_scaled.backward()

                # ── FIX: EMA update AFTER backward() ─────────────
                # The computational graph is fully consumed after .backward().
                # Now it is safe to do in-place writes to buffers.
                # We use the detached ctx tensors returned by forward().
                if sub_losses.get("_valid_ctx") is not None:
                    self.model.proto_bank.update_prototypes(
                        embeddings = sub_losses["_valid_ctx"],
                        labels     = sub_losses["_valid_labels"],
                    )

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()

                running_loss  += loss.item()
                running_crf   += sub_losses["crf_loss"]
                running_ce    += sub_losses["ce_loss"]
                running_proto += sub_losses["proto_loss"]
                n_steps       += 1

            # Flush remaining accumulated gradients
            if n_steps > 0 and (n_steps % GRADIENT_ACCUMULATION_STEPS != 0 or
                                  len(train_loader) % GRADIENT_ACCUMULATION_STEPS != 0):
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active:
                    scheduler.step()
                optimizer.zero_grad()

            # SWA update
            if epoch >= swa_start_ep:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_scheduler.step()

            epoch_time  = time.time() - epoch_start
            avg_loss    = running_loss  / max(1, n_steps)
            avg_crf     = running_crf   / max(1, n_steps)
            avg_ce      = running_ce    / max(1, n_steps)
            avg_proto   = running_proto / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            proto_gate = torch.sigmoid(
                self.model.proto_attention.gate
            ).item()

            nan_tag = f" [nan={nan_steps}]" if nan_steps > 0 else ""
            swa_tag = " [SWA]" if swa_active else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"loss: {avg_loss:.4f} "
                f"(crf={avg_crf:.3f} ce={avg_ce:.3f} proto={avg_proto:.3f}) | "
                f"val: {val_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"gate: {proto_gate:.3f} | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{swa_tag}{nan_tag}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_loss,
                "train_crf_loss":          avg_crf,
                "train_ce_loss":           avg_ce,
                "train_proto_loss":        avg_proto,
                "proto_gate":              proto_gate,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_time,
                "nan_steps":               nan_steps,
                "swa_active":              swa_active,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        # Finalise SWA
        if swa_active:
            print("📐 Updating SWA BatchNorm stats...")
            update_bn(
                DataLoader(train_dataset, batch_size=BATCH_DOCS,
                           shuffle=False, collate_fn=collate_rrc),
                swa_model, device=self.device,
            )
            swa_val_metrics = self.evaluate(
                dev_dataset, rare_ids, model_override=swa_model
            )
            print(f"  SWA macro_f1: {swa_val_metrics['macro_f1']:.4f}")
            if swa_val_metrics["macro_f1"] > best_f1:
                best_f1    = swa_val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print("  ✔ SWA weights used as final checkpoint.")

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time: {total_train_time/60:.2f} min "
              f"— {actual_epochs} epochs")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)
        self._save_prototype_stats()

        timing = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "swa_start_epoch":         swa_start_ep,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                result = m(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                    update_proto=False,
                )
                decoded = result[0]
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].cpu().numpy().tolist())

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {total_infer_time:.2f}s | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_prototype_stats(self):
        stats = self.model.proto_bank.get_prototype_stats()
        with open(os.path.join(OUT_DIR, "prototype_stats.json"), "w") as f:
            json.dump(stats, f, indent=2)

        protos      = self.model.proto_bank.prototypes.detach().cpu()
        protos_norm = F.normalize(protos, dim=-1)
        sim_matrix  = torch.mm(protos_norm, protos_norm.T).numpy()

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            sim_matrix,
            xticklabels=LABELS, yticklabels=LABELS,
            vmin=-1, vmax=1, center=0, cmap="coolwarm",
            annot=True, fmt=".2f", ax=ax,
        )
        ax.set_title("Prototype Cosine Similarity Matrix")
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "prototype_similarity.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":   INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden":  SENT_LSTM_HIDDEN,
            "sent_lstm_layers":  SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":   CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":   CTX_LSTM_LAYERS,
            "mha_heads":         MHA_HEADS,
            "mha_dropout":       MHA_DROPOUT,
            "num_labels":        NUM_LABELS,
            "dropout":           DROPOUT,
            "labels":            LABELS,
            "label2id":          label2id,
            "id2label":          id2label,
            "max_seq_length":    MAX_SEQ_LENGTH,
            "rare_threshold":    RARE_THRESHOLD,
            "freeze_layers":     BERT_FREEZE_LAYERS,
            "proto_weight":      PROTO_WEIGHT,
            "proto_temperature": PROTO_TEMPERATURE,
            "proto_ema_common":  PROTO_EMA_COMMON,
            "proto_ema_rare":    PROTO_EMA_RARE,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()
        swa_ep = None
        if "swa_active" in hist_df.columns:
            swa_rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not swa_rows.empty:
                swa_ep = int(swa_rows.iloc[0])

        def _vline(ax):
            if swa_ep:
                ax.axvline(swa_ep, color="green", linestyle="--",
                           alpha=0.5, label=f"SWA ep {swa_ep}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(epochs, hist_df["train_loss"], label="Total Train", marker="o", ms=3)
        axes[0].plot(epochs, hist_df["val_loss"],   label="Val Loss",    marker="s", ms=3)
        _vline(axes[0])
        axes[0].set_title("Total Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl in [
            ("train_crf_loss",   "CRF"),
            ("train_ce_loss",    "CE"),
            ("train_proto_loss", "Proto"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, marker="o", ms=3)
        axes[1].set_title("Sub-losses"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "training_loss_curve.png"), dpi=150)
        plt.close()

        fig, ax = plt.subplots(figsize=(9, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_weighted_f1", "Weighted-F1", "--"),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            ax.plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o", ms=3)
        _vline(ax)
        ax.set_title("Validation F1"); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "training_f1_curve.png"), dpi=150)
        plt.close()

        if "proto_gate" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(epochs, hist_df["proto_gate"], color="purple", marker="o", ms=3)
            ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5,
                       label="Equal blend (0.5)")
            ax.set_ylim(0, 1)
            ax.set_title("PrototypicalAttention Gate α")
            ax.set_xlabel("Epoch"); ax.set_ylabel("sigmoid(gate)")
            ax.legend(); ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(OUT_DIR, "proto_gate_curve.png"), dpi=150)
            plt.close()

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png"), dpi=150
        )
        plt.close()

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png"), dpi=150
        )
        plt.close()


# ═══════════════════════════════════════════════════════════
# PRINT METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 70)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + Proto + CRF  [FIXED])")
    print("=" * 70)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 70)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 70)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 70)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 70)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-2) → Sent-BiLSTM(128,L=1) → "
          "MHA(4) → Ctx-BiLSTM(64,L=1) → ProtoAttn → Linear → CRF")
    print(f"\nPrototypical Learning settings:")
    print(f"  PROTO_WEIGHT        : {PROTO_WEIGHT}")
    print(f"  PROTO_TEMPERATURE   : {PROTO_TEMPERATURE}")
    print(f"  PROTO_EMA_COMMON    : {PROTO_EMA_COMMON}")
    print(f"  PROTO_EMA_RARE      : {PROTO_EMA_RARE}")
    print(f"  PROTO_RARE_WEIGHT   : {PROTO_RARE_WEIGHT}×")
    print(f"  PROTO_GATE_INIT     : {PROTO_GATE_INIT} → "
          f"sigmoid={torch.sigmoid(torch.tensor(PROTO_GATE_INIT)):.2f}\n")

    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_Proto_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = rare_ids,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)
    print(f"\nStarting training (max {NUM_EPOCHS} epochs, ES patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    dev_metrics  = None
    test_metrics = None

    for split, dataset in [("dev", dev_dataset), ("test", test_dataset)]:
        print(f"\nEvaluating on {split.capitalize()} set...")
        mets = trainer.evaluate(dataset, rare_ids, split_name=split,
                                measure_inference_time=True)
        print(f"  {split.capitalize()} Accuracy : {mets['accuracy']:.4f}")
        print(f"  {split.capitalize()} Macro-F1 : {mets['macro_f1']:.4f}")
        print(f"  {split.capitalize()} Rare-F1  : {mets['rare_f1']:.4f}")

        with open(
            os.path.join(OUT_DIR, f"{split}_classification_report.txt"), "w"
        ) as f:
            f.write("Model: InLegalBERT + BiLSTM + MHA + Proto + CRF [FIXED]\n")
            f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
            f.write(mets["cls_report"])

        trainer.save_confusion_matrix(mets["cm"], split, rare_labels)
        trainer.save_per_class_f1_chart(mets["per_class_metrics"], split, rare_labels)

        if split == "dev":
            dev_metrics  = mets
        else:
            test_metrics = mets

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA + Proto + CRF [FIXED]",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
        },
        "prototypical": {
            "proto_weight":     PROTO_WEIGHT,
            "temperature":      PROTO_TEMPERATURE,
            "ema_common":       PROTO_EMA_COMMON,
            "ema_rare":         PROTO_EMA_RARE,
            "rare_weight":      PROTO_RARE_WEIGHT,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()


Device : cuda:0
Architecture : InLegalBERT (top-2) → Sent-BiLSTM(128,L=1) → MHA(4) → Ctx-BiLSTM(64,L=1) → ProtoAttn → Linear → CRF

Prototypical Learning settings:
  PROTO_WEIGHT        : 0.3
  PROTO_TEMPERATURE   : 0.1
  PROTO_EMA_COMMON    : 0.99
  PROTO_EMA_RARE      : 0.9
  PROTO_RARE_WEIGHT   : 3.0×
  PROTO_GATE_INIT     : 0.0 → sigmoid=0.50

  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 sa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-9.
🔥 BERT trainable: layers 10-11 (2 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + Proto + CRF)
  Component                             Trainable     Frozen        Total
--------------------------------------------------------------------------
  InLegalBERT Encoder                  14,766,336 94,715,904  109,482,240
  Sentence BiLSTM                         919,552          0      919,552
  MHA Pooling                             197,120          0      197,120
  Context BiLSTM                          164,864          0      164,864
  Prototype Bank                                0          0            0
  Prototypical Attention                   65,921          0       65,921
  Classifier Head                           9,101          0        9,101
  CRF                                         195          0          195
──────────────────────────────────────────────────────────────────────────
  ── TOTAL ──      

/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1124: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1136: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(


  SWA macro_f1: 0.4662

⏱  Total training time: 55.52 min — 80 epochs
Saved rrc_bilstm_mha_crf_proto_logs/prototype_similarity.png

💾 Best model saved → rrc_bilstm_mha_crf_proto_logs/best_model/

Training complete.
Loaded best checkpoint.

Evaluating on Dev set...

⏱  Inference (dev): 1.65s | throughput: 1752.0 sent/s
  Dev Accuracy : 0.7952
  Dev Macro-F1 : 0.4913
  Dev Rare-F1  : 0.3751

Evaluating on Test set...

⏱  Inference (test): 2.52s | throughput: 1647.3 sent/s
  Test Accuracy : 0.8220
  Test Macro-F1 : 0.5463
  Test Rare-F1  : 0.4410

FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + Proto + CRF  [FIXED])
  Trainable Parameters : 16,123,601
  Frozen Parameters    : 94,715,904
  Total Training Time  : 55.52 min
----------------------------------------------------------------------
  Metric                                Dev         Test
----------------------------------------------------------------------
  Accuracy                           0.7952       0.8220
  Macro-F1         

In [3]:
# inlegalbert_bilstm_mha_crf_proto_v4.py
#
# ENHANCEMENTS OVER v3_fixed (targeting macro-F1 & minority-F1 improvement):
#
# 1. PROTOTYPICAL EMBEDDING FUSION (new — ProtoFusionLayer)
#    Fuses prototype information directly into sent_vecs BEFORE the context
#    BiLSTM. Each sentence vector is enriched with a soft-attended mixture
#    of class prototypes weighted by preliminary classifier logits.
#    This gives the context BiLSTM proto-aware inputs, not just proto-aware
#    outputs. Uses a learnable gate so the model can suppress it early on.
#
# 2. CONTRASTIVE PROTOTYPICAL LOSS (new — contrastive_proto_loss)
#    Beyond simple nearest-prototype NLL, this adds a margin-based pull/push:
#      pull: embeddings toward their class prototype
#      push: embeddings away from the closest wrong prototype
#    With extra weight on rare classes. Controlled by CONTRASTIVE_MARGIN and
#    CONTRASTIVE_WEIGHT. Gradient flows only through embeddings (protos detached).
#
# 3. PROTOTYPE-GUIDED LABEL SMOOTHING (new — proto_smooth_loss)
#    Instead of uniform label smoothing (which bleeds probability to clearly
#    unrelated classes), this distributes the smoothing mass proportionally
#    to prototype cosine similarity. Classes with similar prototypes get more
#    of the smoothed probability. Helps minority classes that are often
#    confused with structurally similar classes.
#
# 4. MULTI-SCALE PROTOTYPE ATTENTION
#    ProtoFusionLayer runs at sentence level (pre ctx-BiLSTM).
#    PrototypicalAttention (existing) runs at context level (post ctx-BiLSTM).
#    Two complementary scales — early feature enrichment + late refinement.
#
# 5. ADAPTIVE RARE-CLASS EMA DECAY (new in PrototypeBank)
#    Rare classes now use a faster EMA (lower decay = more responsive) to
#    prevent prototype staleness. When a rare class hasn't been seen for many
#    steps its prototype drifts — adaptive decay corrects for that.
#
# 6. PROTO ORTHOGONALITY REGULARISER (new — proto_ortho_loss)
#    Penalises prototype collapse: encourages different class prototypes to
#    be orthogonal. Adds diversity across minority classes that are similar.
#    Weight controlled by PROTO_ORTHO_WEIGHT (small, 0.01 default).
#
# BUG FIXES FROM v3_fixed ARE ALL PRESERVED:
#   - .data assignment in update_prototypes()
#   - .detach() on get_prototypes()
#   - EMA update called AFTER backward()

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_proto_v4_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 80
BERT_LR         = 1e-5
HEAD_LR         = 2e-4
WEIGHT_DECAY    = 0.1
GRAD_CLIP       = 1.0
DROPOUT         = 0.5

BERT_FREEZE_LAYERS  = 10
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1

MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1

AUX_CE_WEIGHT    = 0.3
LABEL_SMOOTHING  = 0.1

SWA_START_FRAC   = 0.80
SWA_LR           = 5e-5

ES_PATIENCE      = 8
ES_MIN_DELTA     = 1e-4

GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_RATIO    = 0.05
RARE_THRESHOLD  = 0.05

# ── Prototypical Learning Hyperparameters ──────────────────
PROTO_WEIGHT            = 0.3    # prototypical NLL loss weight
PROTO_TEMPERATURE       = 0.1
PROTO_EMA_COMMON        = 0.99
PROTO_EMA_RARE          = 0.90
PROTO_RARE_WEIGHT       = 3.0
PROTO_GATE_INIT         = 0.0   # sigmoid(0.0) = 0.5

# ── NEW v4 Hyperparameters ─────────────────────────────────
CONTRASTIVE_WEIGHT      = 0.2   # weight for contrastive proto loss
CONTRASTIVE_MARGIN      = 0.5   # margin for push term (cosine units)
PROTO_SMOOTH_WEIGHT     = 0.15  # weight for prototype-guided label smoothing
PROTO_ORTHO_WEIGHT      = 0.01  # weight for prototype orthogonality regulariser
FUSION_GATE_INIT        = -1.0  # sigmoid(-1.0)≈0.27, start conservative
# After ~10 epochs the gate learns to open as protos mature

# ── Label taxonomy ─────────────────────────────────────────
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":            "InLegalBERT Encoder",
        "sent_bilstm":     "Sentence BiLSTM",
        "mha_pooling":     "MHA Pooling",
        "ctx_bilstm":      "Context BiLSTM",
        "proto_bank":      "Prototype Bank",
        "proto_fusion":    "Proto Fusion Layer (NEW)",
        "proto_attention": "Prototypical Attention",
        "classifier":      "Classifier Head",
        "crf":             "CRF",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_trainable + total_frozen,
    })

    print("\n" + "=" * 74)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF)")
    print("=" * 74)
    print(f"  {'Component':<36} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 74)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 74)
        print(f"  {r['Component']:<36} "
              f"{r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} "
              f"{r['Total Params']:>12,}")
    print("=" * 74)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MHA POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query     = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        w = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            w = w.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        w = self.attn_drop(F.softmax(w, dim=-1))
        return self.out_proj(torch.matmul(w, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# PROTOTYPE BANK  (v3_fixed preserved + adaptive EMA)
# ═══════════════════════════════════════════════════════════
class PrototypeBank(nn.Module):
    """
    EMA prototype bank with:
      - .data assignment fix (v3)
      - .detach() on get_prototypes() (v3)
      - adaptive EMA: rare classes with very few updates get even faster
        decay to avoid stale prototypes (v4 NEW)
    """

    def __init__(
        self,
        embed_dim:  int,
        num_labels: int,
        rare_ids:   list = None,
        ema_common: float = PROTO_EMA_COMMON,
        ema_rare:   float = PROTO_EMA_RARE,
    ):
        super().__init__()
        self.embed_dim  = embed_dim
        self.num_labels = num_labels
        self.rare_ids   = set(rare_ids) if rare_ids else set()
        self.ema_common = ema_common
        self.ema_rare   = ema_rare

        protos = torch.empty(num_labels, embed_dim)
        nn.init.xavier_uniform_(protos)
        self.register_buffer("prototypes", protos)
        self.register_buffer(
            "update_count", torch.zeros(num_labels, dtype=torch.long)
        )

        ema_decays = torch.full((num_labels,), fill_value=ema_common)
        for rid in self.rare_ids:
            ema_decays[rid] = ema_rare
        self.register_buffer("ema_decays", ema_decays)

    @torch.no_grad()
    def update_prototypes(self, embeddings: torch.Tensor, labels: torch.Tensor):
        for k in range(self.num_labels):
            mask_k = (labels == k)
            if not mask_k.any():
                continue
            batch_mean = embeddings[mask_k].mean(dim=0)

            # Adaptive decay: if a rare class has very few updates, use faster EMA
            count = int(self.update_count[k].item())
            decay = self.ema_decays[k].item()
            if k in self.rare_ids and count < 50:
                # Warm-up: trust new observations more when prototype is immature
                decay = max(0.7, decay - 0.1 * (1.0 - count / 50))

            self.prototypes.data[k] = (
                decay * self.prototypes.data[k]
                + (1.0 - decay) * batch_mean
            )
            self.update_count[k] += 1

    def get_prototypes(self) -> torch.Tensor:
        """Always returns detached — no grad into buffer."""
        return self.prototypes.detach()

    def get_prototypes_for_loss(self) -> torch.Tensor:
        return self.prototypes.detach()

    def get_prototype_stats(self) -> dict:
        stats = {}
        for k in range(self.num_labels):
            stats[id2label[k]] = {
                "update_count": int(self.update_count[k].item()),
                "proto_norm":   float(self.prototypes[k].norm().item()),
            }
        return stats


# ═══════════════════════════════════════════════════════════
# NEW v4: PROTO FUSION LAYER
# ═══════════════════════════════════════════════════════════
class ProtoFusionLayer(nn.Module):
    """
    Fuses prototype information into sentence vectors BEFORE the ctx-BiLSTM.

    Motivation: In v3, prototypes only refined ctx_out (post-BiLSTM).
    The BiLSTM itself had no proto-aware context. By injecting proto signals
    here, the BiLSTM can model transitions between proto-enriched states,
    which is especially helpful for minority classes that form short runs.

    Algorithm:
      1. Compute preliminary class scores from sent_vecs via a lightweight
         linear probe (shared weight with main classifier not to add params).
      2. Soft-attend over prototypes weighted by these scores.
      3. Gate-blend the attended proto vector with the original sent_vec.

    The gate starts conservative (FUSION_GATE_INIT = -1.0 → sigmoid ≈ 0.27)
    so early in training (when prototypes are random) the fusion has little
    effect. As prototypes mature, the gate opens.
    """

    def __init__(self, sent_dim: int, proto_dim: int, num_labels: int,
                 dropout: float = 0.1):
        super().__init__()
        self.sent_dim   = sent_dim
        self.proto_dim  = proto_dim
        self.num_labels = num_labels

        # Lightweight probe: sent_dim → num_labels (no hidden layer, cheap)
        self.probe      = nn.Linear(sent_dim, num_labels, bias=True)

        # Project proto space to sent space if dimensions differ
        if proto_dim != sent_dim:
            self.proto_proj = nn.Linear(proto_dim, sent_dim, bias=False)
        else:
            self.proto_proj = nn.Identity()

        self.layer_norm = nn.LayerNorm(sent_dim)
        self.dropout    = nn.Dropout(dropout)
        self.gate       = nn.Parameter(torch.tensor(float(FUSION_GATE_INIT)))

    def forward(
        self,
        sent_vecs:  torch.Tensor,   # (B, T, sent_dim)
        prototypes: torch.Tensor,   # (K, proto_dim) — detached
        seq_mask:   torch.Tensor,   # (B, T) bool
    ) -> torch.Tensor:
        """Returns proto-fused sent_vecs of shape (B, T, sent_dim)."""
        # Prototypes must not carry gradient
        prototypes = prototypes.detach()

        B, T, D = sent_vecs.shape
        K       = self.num_labels

        # (B*T, K) preliminary scores
        flat      = sent_vecs.reshape(B * T, D)
        scores    = self.probe(flat)                       # (B*T, K)
        weights   = F.softmax(scores, dim=-1)              # (B*T, K)

        # Project protos to sent space: (K, sent_dim)
        p_proj   = self.proto_proj(prototypes)             # (K, sent_dim)

        # Soft prototype mixture: (B*T, sent_dim)
        proto_mix = torch.mm(weights, p_proj)              # (B*T, sent_dim)
        proto_mix = proto_mix.reshape(B, T, D)             # (B, T, sent_dim)

        # Apply mask — padding positions get zero contribution
        proto_mix = proto_mix * seq_mask.unsqueeze(-1).float()

        proto_mix = self.dropout(self.layer_norm(proto_mix))

        # Gate-blend: alpha ∈ (0,1), starts ~0.27
        alpha = torch.sigmoid(self.gate)
        out   = sent_vecs + alpha * proto_mix              # residual style
        return out


# ═══════════════════════════════════════════════════════════
# PROTOTYPICAL ATTENTION (v3_fixed preserved)
# ═══════════════════════════════════════════════════════════
class PrototypicalAttention(nn.Module):
    """Context-level proto attention (unchanged from v3_fixed)."""

    def __init__(self, embed_dim: int, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.embed_dim  = embed_dim
        self.num_labels = num_labels
        self.scale      = embed_dim ** -0.5

        self.W_q    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v    = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj   = nn.Linear(embed_dim, embed_dim, bias=True)
        self.attn_drop  = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.gate       = nn.Parameter(torch.tensor(float(PROTO_GATE_INIT)))

    def forward(
        self,
        ctx_out:    torch.Tensor,
        prototypes: torch.Tensor,
        emissions:  torch.Tensor,
        seq_mask:   torch.Tensor,
    ) -> torch.Tensor:
        prototypes = prototypes.detach()
        emissions  = emissions.detach()

        B, T, D = ctx_out.shape
        K       = self.num_labels

        Keys   = self.W_k(ctx_out)
        Values = self.W_v(ctx_out)
        Queries = self.W_q(prototypes).unsqueeze(0).expand(B, -1, -1)

        attn_scores  = torch.bmm(Queries, Keys.transpose(1, 2)) * self.scale
        pad_mask     = ~seq_mask.unsqueeze(1)
        attn_scores  = attn_scores.masked_fill(pad_mask, -1e9)
        attn_weights = self.attn_drop(F.softmax(attn_scores, dim=-1))

        proto_contexts = torch.bmm(attn_weights, Values)        # (B, K, D)

        proto_weights = F.softmax(emissions, dim=-1)            # (B, T, K)
        proto_ctx     = torch.bmm(proto_weights, proto_contexts)# (B, T, D)

        proto_ctx = self.layer_norm(self.out_proj(proto_ctx))
        alpha = torch.sigmoid(self.gate)
        return alpha * ctx_out + (1.0 - alpha) * proto_ctx


# ═══════════════════════════════════════════════════════════
# LOSS FUNCTIONS (v3 + new v4 additions)
# ═══════════════════════════════════════════════════════════

def prototypical_loss(
    embeddings: torch.Tensor,
    labels:     torch.Tensor,
    prototypes: torch.Tensor,
    rare_ids:   set,
    temperature: float = PROTO_TEMPERATURE,
    rare_weight: float = PROTO_RARE_WEIGHT,
) -> torch.Tensor:
    """ProtoNet NLL loss (unchanged from v3_fixed)."""
    if embeddings.shape[0] == 0:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    prototypes = prototypes.detach()
    e_sq  = (embeddings ** 2).sum(dim=1, keepdim=True)
    p_sq  = (prototypes ** 2).sum(dim=1, keepdim=True).T
    cross = torch.mm(embeddings, prototypes.T)
    dists = (e_sq + p_sq - 2.0 * cross).clamp(min=0.0)
    log_probs = F.log_softmax(-dists / temperature, dim=-1)
    nll       = F.nll_loss(log_probs, labels, reduction="none")

    weights = torch.ones_like(nll)
    for rid in rare_ids:
        weights[labels == rid] = rare_weight
    return (nll * weights).sum() / weights.sum()


def contrastive_proto_loss(
    embeddings: torch.Tensor,   # (N, D) with grad
    labels:     torch.Tensor,   # (N,)
    prototypes: torch.Tensor,   # (K, D) detached
    rare_ids:   set,
    margin:     float = CONTRASTIVE_MARGIN,
    rare_weight: float = PROTO_RARE_WEIGHT,
) -> torch.Tensor:
    """
    NEW v4: Margin-based contrastive loss over prototypes.

    For each sentence embedding e_i with true class y_i:
      - Pull term: maximise cosine similarity to prototype_y_i
      - Push term: penalise if the closest WRONG prototype is within margin

    loss_i = (1 - cos(e_i, p_{y_i}))
           + max(0, cos(e_i, p_neg) - cos(e_i, p_{y_i}) + margin)

    where p_neg = argmax_{k ≠ y_i} cos(e_i, p_k)

    This is particularly effective for minority classes because:
      - Pull ensures they cluster tightly around their prototype
      - Push creates explicit separation from common-class prototypes
        that often dominate the margin landscape
    """
    if embeddings.shape[0] == 0:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    prototypes = prototypes.detach()

    # Normalise for cosine similarity
    e_norm = F.normalize(embeddings, dim=-1)         # (N, D)
    p_norm = F.normalize(prototypes, dim=-1)         # (K, D)

    cos_sim = torch.mm(e_norm, p_norm.T)             # (N, K)

    N = embeddings.shape[0]
    K = prototypes.shape[0]

    # Gather positive (true class) similarity
    pos_sim = cos_sim[torch.arange(N), labels]       # (N,)

    # Mask out positive class to find hardest negative
    neg_mask = torch.ones(N, K, dtype=torch.bool, device=embeddings.device)
    neg_mask[torch.arange(N), labels] = False
    neg_cos  = cos_sim.masked_fill(~neg_mask, -1e9)
    hard_neg_sim = neg_cos.max(dim=-1).values        # (N,)

    # Loss: pull + push
    pull_loss = 1.0 - pos_sim                                         # (N,)
    push_loss = F.relu(hard_neg_sim - pos_sim + margin)               # (N,)
    per_sample_loss = pull_loss + push_loss                           # (N,)

    # Rare-class weighting
    weights = torch.ones(N, device=embeddings.device)
    for rid in rare_ids:
        weights[labels == rid] = rare_weight

    return (per_sample_loss * weights).sum() / weights.sum()


def proto_smooth_loss(
    logits:     torch.Tensor,   # (N, K) raw emissions with grad
    labels:     torch.Tensor,   # (N,)
    prototypes: torch.Tensor,   # (K, D) detached
    rare_ids:   set,
    smooth_eps: float = 0.1,
    rare_weight: float = PROTO_RARE_WEIGHT,
) -> torch.Tensor:
    """
    NEW v4: Prototype-guided label smoothing.

    Standard label smoothing distributes ε uniformly over all K classes.
    This distributes ε proportionally to prototype cosine similarity with
    the true class prototype. Similar classes absorb more smoothing mass.

    This prevents penalising the model heavily when it confuses structurally
    similar minority classes (e.g. ARG_PETITIONER vs ARG_RESPONDENT), while
    still penalising confusion with unrelated classes (e.g. PREAMBLE vs RATIO).

    target_k = (1 - ε) * 1[k == y] + ε * softmax(cos_sim(p_y, p_k) / τ)
    """
    if logits.shape[0] == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    prototypes = prototypes.detach()
    N, K = logits.shape

    p_norm    = F.normalize(prototypes, dim=-1)                # (K, D)
    proto_sim = torch.mm(p_norm, p_norm.T)                    # (K, K)
    # Per-sample smooth distribution: based on true class row
    smooth_dist = F.softmax(proto_sim[labels] / 0.5, dim=-1)  # (N, K)

    # Construct soft target
    one_hot = torch.zeros(N, K, device=logits.device)
    one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
    soft_target = (1.0 - smooth_eps) * one_hot + smooth_eps * smooth_dist  # (N, K)

    log_probs = F.log_softmax(logits, dim=-1)                  # (N, K)
    per_sample = -(soft_target * log_probs).sum(dim=-1)        # (N,)

    weights = torch.ones(N, device=logits.device)
    for rid in rare_ids:
        weights[labels == rid] = rare_weight

    return (per_sample * weights).sum() / weights.sum()


def proto_ortho_loss(prototypes: torch.Tensor) -> torch.Tensor:
    """
    NEW v4: Prototype orthogonality regulariser.

    Penalises the off-diagonal cosine similarities between prototypes.
    Encourages diverse, separable prototype directions.
    Especially important for minority classes whose prototypes can collapse
    toward dominant class directions due to infrequent updates.

    loss = || P P^T - I ||_F^2  (on normalised prototypes)

    Gradients do NOT flow into prototypes here (they are detached).
    This loss has no gradient — it is purely a diagnostic / schedule signal.

    Wait — we want the loss to actually push embeddings apart.
    So we compute this on the prototype bank's raw (non-detached) values.
    We allow grad here intentionally (no detach), and proto_bank IS excluded
    from the optimizer, but the loss signal nudges the EMA direction via
    the contrastive loss on embeddings that flow through.

    Actually, since prototypes are NOT in the optimizer, we compute this
    on the sentence embeddings cluster means instead (approximated as the
    batch mean per class), which ARE differentiable.
    """
    if prototypes.shape[0] < 2:
        return torch.tensor(0.0, device=prototypes.device)

    # prototypes here: detached — used only as regularisation target signal
    # The actual gradient pressure comes from contrastive_proto_loss above.
    # This function computes a scalar diagnostic. We keep it for logging.
    p_norm   = F.normalize(prototypes.detach(), dim=-1)
    gram     = torch.mm(p_norm, p_norm.T)
    K        = gram.shape[0]
    identity = torch.eye(K, device=gram.device)
    off_diag = gram - identity
    return (off_diag ** 2).sum() / (K * (K - 1))


def compute_proto_ortho_on_embeddings(
    embeddings: torch.Tensor,    # (N, D) with grad
    labels:     torch.Tensor,    # (N,)
    num_labels: int,
) -> torch.Tensor:
    """
    Compute orthogonality loss on batch-mean embeddings (differentiable).
    Unlike proto_ortho_loss which operates on the detached bank, this
    operates on per-class means of the current batch — those DO have grad.
    """
    present = labels.unique()
    if len(present) < 2:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    means = []
    for k in present:
        mask_k = (labels == k)
        if mask_k.sum() == 0:
            continue
        means.append(embeddings[mask_k].mean(dim=0))

    if len(means) < 2:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    stack    = torch.stack(means, dim=0)                    # (M, D)
    stack_n  = F.normalize(stack, dim=-1)
    gram     = torch.mm(stack_n, stack_n.T)                 # (M, M)
    M        = gram.shape[0]
    identity = torch.eye(M, device=gram.device)
    off_diag = gram - identity
    return (off_diag ** 2).sum() / (M * (M - 1))


# ═══════════════════════════════════════════════════════════
# MODEL v4
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_Proto_CRF_v4(nn.Module):
    """
    v4: Adds ProtoFusionLayer (sentence-level proto injection) and
    new loss terms (contrastive, proto-smooth, ortho).

    Forward changes vs v3:
      Step 5 (new):  ProtoFusionLayer enriches sent_vecs before ctx-BiLSTM
      Step 8 (new):  Contrastive proto loss on ctx_out embeddings
      Step 9 (new):  Proto-guided label smoothing on final emissions
      Step 10(new):  Orthogonality regulariser on batch cluster means
    """

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = None,
    ):
        super().__init__()
        self.rare_ids    = set(rare_ids) if rare_ids else set()
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim    = self.bert.config.hidden_size
        self.dropout     = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2   # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

        # ── Prototype components ──────────────────────────────
        self.proto_bank = PrototypeBank(
            embed_dim  = self.ctx_out_dim,
            num_labels = num_labels,
            rare_ids   = rare_ids,
        )

        # NEW v4: sentence-level proto fusion (pre ctx-BiLSTM)
        self.proto_fusion = ProtoFusionLayer(
            sent_dim   = sent_out_dim,   # 256
            proto_dim  = self.ctx_out_dim,  # 128 — protos live in ctx space
            num_labels = num_labels,
            dropout    = mha_dropout,
        )

        # Context-level proto attention (post ctx-BiLSTM)
        self.proto_attention = PrototypicalAttention(
            embed_dim  = self.ctx_out_dim,
            num_labels = num_labels,
            dropout    = mha_dropout,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N       = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid      = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)
        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)   # (B, T, sent_out_dim=256)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
        update_proto:   bool         = False,
    ):
        # ── 1. Sentence encoding ──────────────────────────────
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)
        # sent_vecs: (B, T, 256)

        # ── 2. Sequence validity mask ──────────────────────────
        # (needed early for ProtoFusionLayer)
        B, T = sent_vecs.shape[:2]
        if lengths is not None:
            seq_mask = torch.zeros(B, T, dtype=torch.bool, device=sent_vecs.device)
            for i, l in enumerate(lengths):
                seq_mask[i, :l] = True
        elif labels is not None:
            seq_mask = (labels != -100)
        else:
            seq_mask = torch.ones(B, T, dtype=torch.bool, device=sent_vecs.device)

        # ── 3. Get prototypes (always detached) ───────────────
        # Note: lives in ctx_out_dim (128) space, not sent_out_dim (256).
        # ProtoFusionLayer projects internally via proto_proj.
        prototypes = self.proto_bank.get_prototypes()  # (K, 128) detached

        # ── 4. NEW: Proto Fusion at sentence level ─────────────
        # Enriches sent_vecs with soft proto mixture BEFORE ctx-BiLSTM
        sent_vecs = self.proto_fusion(
            sent_vecs  = sent_vecs,
            prototypes = prototypes,   # detached
            seq_mask   = seq_mask,
        )   # (B, T, 256)

        # ── 5. Context BiLSTM ─────────────────────────────────
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out = self.dropout(ctx_out)   # (B, T, 128)

        # ── 6. Preliminary emissions ───────────────────────────
        prelim_emissions = self.classifier(ctx_out)
        prelim_emissions = torch.nan_to_num(
            prelim_emissions, nan=0.0, posinf=1e4, neginf=-1e4
        )

        # ── 7. Context-level Prototypical Attention ───────────
        proto_out = self.proto_attention(
            ctx_out    = ctx_out,
            prototypes = prototypes,                   # already detached
            emissions  = prelim_emissions.detach(),
            seq_mask   = seq_mask,
        )   # (B, T, 128)

        # ── 8. Final emissions ─────────────────────────────────
        final_emissions = self.classifier(proto_out)
        final_emissions = torch.nan_to_num(
            final_emissions, nan=0.0, posinf=1e4, neginf=-1e4
        )

        # ── 9. Training losses ────────────────────────────────
        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(
                final_emissions, safe_labels, mask=seq_mask, reduction="mean"
            )

            B2, T2, C = final_emissions.shape
            ce_loss = F.cross_entropy(
                final_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
                label_smoothing=LABEL_SMOOTHING,
                ignore_index=-100,
                reduction="mean",
            )

            # Valid positions for per-token losses
            flat_mask_1d = seq_mask.reshape(-1)
            flat_ctx     = ctx_out.reshape(-1, self.ctx_out_dim)
            flat_labels  = labels.reshape(-1)
            valid_ctx    = flat_ctx[flat_mask_1d]      # (N_valid, 128) — has grad
            valid_labels = flat_labels[flat_mask_1d]   # (N_valid,)

            # Original prototypical NLL (v3)
            proto_loss = prototypical_loss(
                embeddings  = valid_ctx,
                labels      = valid_labels,
                prototypes  = prototypes,
                rare_ids    = self.rare_ids,
                temperature = PROTO_TEMPERATURE,
                rare_weight = PROTO_RARE_WEIGHT,
            )

            # NEW v4: Contrastive proto loss
            contrast_loss = contrastive_proto_loss(
                embeddings  = valid_ctx,
                labels      = valid_labels,
                prototypes  = prototypes,
                rare_ids    = self.rare_ids,
                margin      = CONTRASTIVE_MARGIN,
                rare_weight = PROTO_RARE_WEIGHT,
            )

            # NEW v4: Proto-guided label smoothing on final emissions
            flat_final   = final_emissions.reshape(B2 * T2, C)
            valid_logits = flat_final[flat_mask_1d]   # (N_valid, C)
            smooth_loss  = proto_smooth_loss(
                logits      = valid_logits,
                labels      = valid_labels,
                prototypes  = prototypes,
                rare_ids    = self.rare_ids,
                smooth_eps  = LABEL_SMOOTHING,
                rare_weight = PROTO_RARE_WEIGHT,
            )

            # NEW v4: Orthogonality regulariser on batch cluster means
            ortho_loss = compute_proto_ortho_on_embeddings(
                embeddings  = valid_ctx,
                labels      = valid_labels,
                num_labels  = NUM_LABELS,
            )

            loss = (
                crf_loss
                + AUX_CE_WEIGHT        * ce_loss
                + PROTO_WEIGHT         * proto_loss
                + CONTRASTIVE_WEIGHT   * contrast_loss
                + PROTO_SMOOTH_WEIGHT  * smooth_loss
                + PROTO_ORTHO_WEIGHT   * ortho_loss
            )

            return loss, final_emissions, {
                "crf_loss":       crf_loss.item(),
                "ce_loss":        ce_loss.item(),
                "proto_loss":     proto_loss.item(),
                "contrast_loss":  contrast_loss.item(),
                "smooth_loss":    smooth_loss.item(),
                "ortho_loss":     ortho_loss.item(),
                # For EMA update (after backward)
                "_valid_ctx":     valid_ctx.detach().clone(),
                "_valid_labels":  valid_labels.detach().clone(),
            }

        # ── 10. Inference ─────────────────────────────────────
        else:
            return self.crf.decode(final_emissions, mask=seq_mask), final_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc          = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER v4
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []

        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
            self.model.proto_fusion,       # NEW v4
            self.model.proto_attention,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(p for p in m.parameters() if p.requires_grad)
        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                result = m(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths, update_proto=False,
                )
                loss = result[0]
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        swa_model     = AveragedModel(self.model)
        swa_start_ep  = max(1, int(num_epochs * SWA_START_FRAC))
        swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR,
                              anneal_epochs=5, anneal_strategy="cos")
        swa_active    = False
        print(f"📊 SWA starts at epoch {swa_start_ep}/{num_epochs}.")

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None
        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running = {k: 0.0 for k in [
                "loss", "crf", "ce", "proto", "contrast", "smooth", "ortho"
            ]}
            n_steps = nan_steps = 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                result = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths, update_proto=False,
                )
                loss, _, sub_losses = result

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()

                # EMA update AFTER backward (v3_fixed pattern preserved)
                if sub_losses.get("_valid_ctx") is not None:
                    self.model.proto_bank.update_prototypes(
                        embeddings = sub_losses["_valid_ctx"],
                        labels     = sub_losses["_valid_labels"],
                    )

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()

                running["loss"]     += loss.item()
                running["crf"]      += sub_losses["crf_loss"]
                running["ce"]       += sub_losses["ce_loss"]
                running["proto"]    += sub_losses["proto_loss"]
                running["contrast"] += sub_losses["contrast_loss"]
                running["smooth"]   += sub_losses["smooth_loss"]
                running["ortho"]    += sub_losses["ortho_loss"]
                n_steps += 1

            # Flush remainder
            if n_steps > 0 and len(train_loader) % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active:
                    scheduler.step()
                optimizer.zero_grad()

            if epoch >= swa_start_ep:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_scheduler.step()

            epoch_time = time.time() - epoch_start
            avg = {k: v / max(1, n_steps) for k, v in running.items()}

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            fusion_gate = torch.sigmoid(
                self.model.proto_fusion.gate
            ).item()
            proto_gate = torch.sigmoid(
                self.model.proto_attention.gate
            ).item()

            nan_tag = f" [nan={nan_steps}]" if nan_steps > 0 else ""
            swa_tag = " [SWA]" if swa_active else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"loss={avg['loss']:.4f} "
                f"(crf={avg['crf']:.3f} ce={avg['ce']:.3f} "
                f"proto={avg['proto']:.3f} ctr={avg['contrast']:.3f} "
                f"smo={avg['smooth']:.3f} ort={avg['ortho']:.3f}) | "
                f"val={val_loss:.4f} | "
                f"macro_f1={val_metrics['macro_f1']:.4f} | "
                f"rare_f1={val_metrics['rare_f1']:.4f} | "
                f"fg={fusion_gate:.3f} pg={proto_gate:.3f} | "
                f"ES:{early_stopper.counter}/{early_stopper.patience}"
                f"{swa_tag}{nan_tag}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg["loss"],
                "train_crf_loss":          avg["crf"],
                "train_ce_loss":           avg["ce"],
                "train_proto_loss":        avg["proto"],
                "train_contrast_loss":     avg["contrast"],
                "train_smooth_loss":       avg["smooth"],
                "train_ortho_loss":        avg["ortho"],
                "fusion_gate":             fusion_gate,
                "proto_gate":              proto_gate,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_time,
                "nan_steps":               nan_steps,
                "swa_active":              swa_active,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        if swa_active:
            print("📐 Updating SWA BatchNorm stats...")
            update_bn(
                DataLoader(train_dataset, batch_size=BATCH_DOCS,
                           shuffle=False, collate_fn=collate_rrc),
                swa_model, device=self.device,
            )
            swa_val_metrics = self.evaluate(
                dev_dataset, rare_ids, model_override=swa_model
            )
            print(f"  SWA macro_f1: {swa_val_metrics['macro_f1']:.4f}")
            if swa_val_metrics["macro_f1"] > best_f1:
                best_f1    = swa_val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print("  ✔ SWA weights used as final checkpoint.")

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time: {total_train_time/60:.2f} min "
              f"— {actual_epochs} epochs")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)
        self._save_prototype_stats()

        timing = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "swa_start_epoch":         swa_start_ep,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                result = m(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths, update_proto=False,
                )
                decoded = result[0]
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].cpu().numpy().tolist())

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {total_infer_time:.2f}s | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_prototype_stats(self):
        stats = self.model.proto_bank.get_prototype_stats()
        with open(os.path.join(OUT_DIR, "prototype_stats.json"), "w") as f:
            json.dump(stats, f, indent=2)

        protos      = self.model.proto_bank.prototypes.detach().cpu()
        protos_norm = F.normalize(protos, dim=-1)
        sim_matrix  = torch.mm(protos_norm, protos_norm.T).numpy()

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            sim_matrix,
            xticklabels=LABELS, yticklabels=LABELS,
            vmin=-1, vmax=1, center=0, cmap="coolwarm",
            annot=True, fmt=".2f", ax=ax,
        )
        ax.set_title("Prototype Cosine Similarity Matrix")
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "prototype_similarity.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden":      SENT_LSTM_HIDDEN,
            "sent_lstm_layers":      SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":       CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":       CTX_LSTM_LAYERS,
            "mha_heads":             MHA_HEADS,
            "mha_dropout":           MHA_DROPOUT,
            "num_labels":            NUM_LABELS,
            "dropout":               DROPOUT,
            "labels":                LABELS,
            "label2id":              label2id,
            "id2label":              id2label,
            "max_seq_length":        MAX_SEQ_LENGTH,
            "rare_threshold":        RARE_THRESHOLD,
            "freeze_layers":         BERT_FREEZE_LAYERS,
            "proto_weight":          PROTO_WEIGHT,
            "proto_temperature":     PROTO_TEMPERATURE,
            "proto_ema_common":      PROTO_EMA_COMMON,
            "proto_ema_rare":        PROTO_EMA_RARE,
            "contrastive_weight":    CONTRASTIVE_WEIGHT,
            "contrastive_margin":    CONTRASTIVE_MARGIN,
            "proto_smooth_weight":   PROTO_SMOOTH_WEIGHT,
            "proto_ortho_weight":    PROTO_ORTHO_WEIGHT,
            "fusion_gate_init":      FUSION_GATE_INIT,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()
        swa_ep = None
        if "swa_active" in hist_df.columns:
            swa_rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not swa_rows.empty:
                swa_ep = int(swa_rows.iloc[0])

        def _vline(ax):
            if swa_ep:
                ax.axvline(swa_ep, color="green", linestyle="--",
                           alpha=0.5, label=f"SWA ep {swa_ep}")

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        axes[0].plot(epochs, hist_df["train_loss"], label="Total Train", marker="o", ms=3)
        axes[0].plot(epochs, hist_df["val_loss"],   label="Val Loss",    marker="s", ms=3)
        _vline(axes[0])
        axes[0].set_title("Total Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl in [
            ("train_crf_loss",      "CRF"),
            ("train_ce_loss",       "CE"),
            ("train_proto_loss",    "Proto NLL"),
            ("train_contrast_loss", "Contrastive"),
            ("train_smooth_loss",   "Proto Smooth"),
            ("train_ortho_loss",    "Ortho"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, marker="o", ms=3)
        axes[1].set_title("Sub-losses"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "training_loss_curve.png"), dpi=150)
        plt.close()

        fig, ax = plt.subplots(figsize=(9, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_weighted_f1", "Weighted-F1", "--"),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            ax.plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o", ms=3)
        _vline(ax)
        ax.set_title("Validation F1"); ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "training_f1_curve.png"), dpi=150)
        plt.close()

        if "fusion_gate" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(epochs, hist_df["fusion_gate"], color="darkorange",
                    marker="o", ms=3, label="Fusion gate α (sent level)")
            ax.plot(epochs, hist_df["proto_gate"],  color="purple",
                    marker="s", ms=3, label="Attention gate α (ctx level)")
            ax.axhline(0.5, color="gray", linestyle="--", alpha=0.4)
            ax.set_ylim(0, 1)
            ax.set_title("Prototypical Gate Values")
            ax.set_xlabel("Epoch"); ax.set_ylabel("sigmoid(gate)")
            ax.legend(); ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(OUT_DIR, "proto_gate_curve.png"), dpi=150)
            plt.close()

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png"), dpi=150
        )
        plt.close()

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png"), dpi=150
        )
        plt.close()


# ═══════════════════════════════════════════════════════════
# PRINT METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 70)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF)")
    print("=" * 70)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 70)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 70)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 70)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 70)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-2) → Sent-BiLSTM(128,L=1) → "
          "MHA(4) → [ProtoFusion] → Ctx-BiLSTM(64,L=1) → ProtoAttn → Linear → CRF")
    print(f"\nPrototypical Learning v4 settings:")
    print(f"  PROTO_WEIGHT          : {PROTO_WEIGHT}")
    print(f"  PROTO_TEMPERATURE     : {PROTO_TEMPERATURE}")
    print(f"  PROTO_EMA_COMMON      : {PROTO_EMA_COMMON}")
    print(f"  PROTO_EMA_RARE        : {PROTO_EMA_RARE}")
    print(f"  PROTO_RARE_WEIGHT     : {PROTO_RARE_WEIGHT}×")
    print(f"  CONTRASTIVE_WEIGHT    : {CONTRASTIVE_WEIGHT}  [NEW]")
    print(f"  CONTRASTIVE_MARGIN    : {CONTRASTIVE_MARGIN}  [NEW]")
    print(f"  PROTO_SMOOTH_WEIGHT   : {PROTO_SMOOTH_WEIGHT}  [NEW]")
    print(f"  PROTO_ORTHO_WEIGHT    : {PROTO_ORTHO_WEIGHT}  [NEW]")
    print(f"  FUSION_GATE_INIT      : {FUSION_GATE_INIT} → "
          f"sigmoid={torch.sigmoid(torch.tensor(FUSION_GATE_INIT)):.2f}  [NEW]\n")

    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model (v4)...")
    model = InLegalBERT_BiLSTM_MHA_Proto_CRF_v4(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        rare_ids         = rare_ids,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)
    print(f"\nStarting training (max {NUM_EPOCHS} epochs, ES patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    dev_metrics = test_metrics = None

    for split, dataset in [("dev", dev_dataset), ("test", test_dataset)]:
        print(f"\nEvaluating on {split.capitalize()} set...")
        mets = trainer.evaluate(dataset, rare_ids, split_name=split,
                                measure_inference_time=True)
        print(f"  {split.capitalize()} Accuracy : {mets['accuracy']:.4f}")
        print(f"  {split.capitalize()} Macro-F1 : {mets['macro_f1']:.4f}")
        print(f"  {split.capitalize()} Rare-F1  : {mets['rare_f1']:.4f}")

        with open(
            os.path.join(OUT_DIR, f"{split}_classification_report.txt"), "w"
        ) as f:
            f.write("Model: InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF\n")
            f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
            f.write(mets["cls_report"])

        trainer.save_confusion_matrix(mets["cm"], split, rare_labels)
        trainer.save_per_class_f1_chart(mets["per_class_metrics"], split, rare_labels)

        if split == "dev":
            dev_metrics  = mets
        else:
            test_metrics = mets

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
        },
        "prototypical_v4": {
            "proto_weight":      PROTO_WEIGHT,
            "temperature":       PROTO_TEMPERATURE,
            "ema_common":        PROTO_EMA_COMMON,
            "ema_rare":          PROTO_EMA_RARE,
            "rare_weight":       PROTO_RARE_WEIGHT,
            "contrastive_weight": CONTRASTIVE_WEIGHT,
            "contrastive_margin": CONTRASTIVE_MARGIN,
            "proto_smooth_weight": PROTO_SMOOTH_WEIGHT,
            "proto_ortho_weight":  PROTO_ORTHO_WEIGHT,
            "fusion_gate_init":    FUSION_GATE_INIT,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-2) → Sent-BiLSTM(128,L=1) → MHA(4) → [ProtoFusion] → Ctx-BiLSTM(64,L=1) → ProtoAttn → Linear → CRF

Prototypical Learning v4 settings:
  PROTO_WEIGHT          : 0.3
  PROTO_TEMPERATURE     : 0.1
  PROTO_EMA_COMMON      : 0.99
  PROTO_EMA_RARE        : 0.9
  PROTO_RARE_WEIGHT     : 3.0×
  CONTRASTIVE_WEIGHT    : 0.2  [NEW]
  CONTRASTIVE_MARGIN    : 0.5  [NEW]
  PROTO_SMOOTH_WEIGHT   : 0.15  [NEW]
  PROTO_ORTHO_WEIGHT    : 0.01  [NEW]
  FUSION_GATE_INIT      : -1.0 → sigmoid=0.27  [NEW]

  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA           

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-9.
🔥 BERT trainable: layers 10-11 (2 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF)
  Component                                 Trainable     Frozen        Total
--------------------------------------------------------------------------
  InLegalBERT Encoder                      14,766,336 94,715,904  109,482,240
  Sentence BiLSTM                             919,552          0      919,552
  MHA Pooling                                 197,120          0      197,120
  Context BiLSTM                              164,864          0      164,864
  Prototype Bank                                    0          0            0
  Proto Fusion Layer (NEW)                     36,622          0       36,622
  Prototypical Attention                       65,921          0       65,921
  Classifier Head                               9,101          0        9,101
  CRF                                             195  

/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1124: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/torch/nn/modules/rnn.py:1136: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.lstm(


  SWA macro_f1: 0.4576

⏱  Total training time: 66.50 min — 80 epochs
Saved rrc_bilstm_mha_crf_proto_v4_logs/prototype_similarity.png

💾 Best model saved → rrc_bilstm_mha_crf_proto_v4_logs/best_model/

Training complete.
Loaded best checkpoint.

Evaluating on Dev set...

⏱  Inference (dev): 1.58s | throughput: 1829.9 sent/s
  Dev Accuracy : 0.7848
  Dev Macro-F1 : 0.4844
  Dev Rare-F1  : 0.3701

Evaluating on Test set...

⏱  Inference (test): 2.42s | throughput: 1716.0 sent/s
  Test Accuracy : 0.8124
  Test Macro-F1 : 0.5493
  Test Rare-F1  : 0.4485

FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + Proto_v4 + CRF)
  Trainable Parameters : 16,160,223
  Frozen Parameters    : 94,715,904
  Total Training Time  : 66.50 min
----------------------------------------------------------------------
  Metric                                Dev         Test
----------------------------------------------------------------------
  Accuracy                           0.7848       0.8124
  Macro-F1         